In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "06-gateway/identity-security/agentic-identity-gcp-lab/notebooks/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


    # 02 · Delegation and token exchange — practice

    **Primer section:** §3.5. Build the delegation chain yourself: exchange, narrow, nest, bind with
    DPoP, and downscope with a Credential Access Boundary.

> **How to use this practice notebook.** Every `____` is a blank you must fill (a name, an
> argument, an expression); a `raise NotImplementedError("fill me")` means "write the body".
> Each exercise ends with `assert` checks — run the cell, and if it is silent you got it right.
> The completed version lives in `notebooks/solutions/`.

In [ ]:
from agentsec.logging_utils import quiet_logs

quiet_logs()

import json

import jwt  # display only: unverified peeks

from agentsec.identity import (
    AgentIdentity,
    AuthorityContext,
    AuthorityMode,
    BindingMismatch,
    BoundaryEvaluator,
    BoundaryRule,
    DPoP,
    ExpiredToken,
    InsufficientScope,
    InvalidAudience,
    LocalRuntimeCA,
    ReplayDetected,
    TokenIssuer,
    UserPrincipal,
    build_boundary,
    jwk_thumbprint,
    public_jwk,
)


def peek(token: str) -> dict:
    return jwt.decode(token, options={"verify_signature": False})

def short(spiffe: str) -> str:
    return spiffe.rsplit("/", 1)[-1]

ORG, PROJECT = "123456789012", "987654321098"
agent = AgentIdentity.for_agent_engine(project_number=PROJECT, location="us-central1", engine_id="support-agent", org_id=ORG)
sub_agent = AgentIdentity.for_agent_engine(project_number=PROJECT, location="us-central1", engine_id="refunds-subagent", org_id=ORG)
ana = UserPrincipal(subject="u-ana", email="ana@customer.example", tenant="acme")

ca = LocalRuntimeCA()
cert = ca.issue(agent)
issuer = TokenIssuer()          # the local STS / authorization server

FRONT_END = "https://app.acme.example"      # audience of the user's login token
TICKETS_MCP = "https://tickets.example/mcp" # canonical URI of a tool server (RFC 8707 resource)
PAYMENTS = "https://payments.example"

## Exercise 1 — perform an RFC 8693 exchange

Mint the user's login token (audience = the front-end, scopes `tickets:read tickets:write`) and
the agent's certificate-bound actor token (audience = the STS), then exchange them for a token for
the tickets MCP server with scope `tickets:read`.

In [ ]:
user_token = issuer.mint(subject=ana.subject, audience=FRONT_END, scope="tickets:read tickets:write", extra={"email": ana.email})
actor_token = issuer.mint_agent_token(cert, audience=issuer.issuer)

resp = issuer.exchange(
    subject_token=user_token, subject_token_audience=FRONT_END,
    actor_token=actor_token, actor_token_audience=issuer.issuer, presented_thumbprint=cert.thumbprint,
    audience=TICKETS_MCP, scope="tickets:read",
)
delegated = resp["access_token"]
claims = issuer.verify(delegated, audience=TICKETS_MCP, presented_thumbprint=cert.thumbprint)

assert claims.subject == "u-ana" and claims.actor == agent.spiffe_id
assert claims.is_delegated and claims.raw["authority"] == "delegated"
assert claims.scopes == {"tickets:read"} and claims.audience == TICKETS_MCP
assert claims.cnf["x5t#S256"] == cert.thumbprint
assert AuthorityContext.from_claims(claims).mode is AuthorityMode.DELEGATED
print(f"delegated token: sub={claims.subject} act={short(claims.actor)} scope={claims.scopes}")

## Exercise 2 — prove the exchange cannot widen scopes

Request `tickets:read payments:refund` and assert that only what the user consented to survives. The cell
records the scope string you send, so the check fails unless the widened request is actually made.

In [ ]:
requested = []   # every scope string this cell sends to the STS, so the check can see what was asked for

def exchange_recorded(**kw):
    requested.append(kw["scope"])
    return issuer.exchange(**kw)

resp2 = exchange_recorded(
    subject_token=user_token, subject_token_audience=FRONT_END,
    actor_token=actor_token, actor_token_audience=issuer.issuer, presented_thumbprint=cert.thumbprint,
    audience=TICKETS_MCP, scope="tickets:read payments:refund",
)
granted = set(resp2["scope"].split())
consented = set(peek(user_token)["scope"].split())
asked = set(requested[-1].split())
assert "payments:refund" in asked and asked - consented, f"request a scope the user never consented to (asked for {asked})"
assert granted == {"tickets:read"}, granted          # the STS dropped what the user never consented to
try:                                                 # and a resource server demanding it rejects the token
    issuer.verify(resp2["access_token"], audience=TICKETS_MCP, presented_thumbprint=cert.thumbprint, required_scopes={"payments:refund"})
    raise AssertionError("the delegated token must not carry payments:refund")
except InsufficientScope:
    pass
print("asked for:", sorted(asked), "→ granted:", sorted(granted))

## Exercise 3 — add a hop to the refunds sub-agent

Exchange the *delegated* token again with the sub-agent as actor, for the payments audience. The
resulting `actor_chain` must list the sub-agent first, then the support agent.

In [ ]:
sub_agent_cert = ca.issue(sub_agent)
sub_actor_token = issuer.mint_agent_token(sub_agent_cert, audience=issuer.issuer)

resp3 = issuer.exchange(
    subject_token=delegated, subject_token_audience=TICKETS_MCP,
    actor_token=sub_actor_token, actor_token_audience=issuer.issuer, presented_thumbprint=sub_agent_cert.thumbprint,
    audience=PAYMENTS, scope="tickets:read",
)
hop2 = issuer.verify(resp3["access_token"], audience=PAYMENTS, presented_thumbprint=sub_agent_cert.thumbprint)

assert hop2.actor_chain == [sub_agent.spiffe_id, agent.spiffe_id]
assert hop2.subject == "u-ana" and hop2.scopes == {"tickets:read"}
assert AuthorityContext.from_claims(hop2).chain == (sub_agent.spiffe_id, agent.spiffe_id)
print("chain:", [short(a) for a in hop2.actor_chain])

## Exercise 4 — write the resource server's checks

Implement `accept(token, audience, required, thumbprint)` returning the error class name on
failure and `"ok"` on success. Use the library verifier; do not decode the JWT yourself.

In [ ]:
def accept(token: str, *, audience: str, required: set[str] | None = None, thumbprint: str | None = None) -> str:
    try:
        issuer.verify(token, audience=audience, required_scopes=required, presented_thumbprint=thumbprint)
        return "ok"
    except (InvalidAudience, InsufficientScope, ExpiredToken, BindingMismatch) as e:
        return type(e).__name__

expired = issuer.mint(subject="u-ana", audience=TICKETS_MCP, ttl=-30)
assert accept(delegated, audience=TICKETS_MCP, thumbprint=cert.thumbprint) == "ok"
assert accept(delegated, audience=PAYMENTS, thumbprint=cert.thumbprint) == "InvalidAudience"
assert accept(delegated, audience=TICKETS_MCP, required={"tickets:write"}, thumbprint=cert.thumbprint) == "InsufficientScope"
assert accept(delegated, audience=TICKETS_MCP) == "BindingMismatch"
assert accept(expired, audience=TICKETS_MCP) == "ExpiredToken"
print("verifier distinguishes audience, scope, binding and expiry failures")

## Exercise 5 — DPoP: bind, prove, replay

Generate a DPoP key, mint a token bound to it, sign a proof for `POST TICKETS_MCP` with `ath`,
verify it once (recording `jti`), then show that the same proof is rejected as a replay.

In [ ]:
key = DPoP.generate_key()
jwk = public_jwk(key)
dpop_token = issuer.mint_dpop_bound_token(subject="u-ana", audience=TICKETS_MCP, dpop_public_jwk=jwk, scope="tickets:read")
proof = DPoP.proof(key, method="POST", url=TICKETS_MCP, access_token=dpop_token)

seen_jti: set[str] = set()
proof_claims = DPoP.verify(proof, method="POST", url=TICKETS_MCP, access_token=dpop_token, seen_jti=seen_jti)
assert proof_claims["jkt"] == jwk_thumbprint(jwk)
issuer.verify(dpop_token, audience=TICKETS_MCP, presented_jkt=proof_claims["jkt"])

try:
    DPoP.verify(proof, method="POST", url=TICKETS_MCP, access_token=dpop_token, seen_jti=seen_jti)
    raise AssertionError("replay must fail")
except ReplayDetected:
    print("replay rejected")

try:
    DPoP.verify(proof, method="POST", url=TICKETS_MCP, access_token=dpop_token + "x")
    raise AssertionError("ath mismatch must fail")
except BindingMismatch:
    print("ath mismatch rejected")

## Exercise 6 — downscope with a Credential Access Boundary

Build a boundary that lets a tool **read** objects under `tenant-a/` in bucket `acme-invoices`
and nothing else, then check four accesses with the evaluator.

In [ ]:
boundary = build_boundary(BoundaryRule(bucket="acme-invoices", prefix="tenant-a/", roles=("roles/storage.objectViewer",)))
rule = boundary.to_json()["accessBoundary"]["accessBoundaryRules"][0]
assert rule["availableResource"] == "//storage.googleapis.com/projects/_/buckets/acme-invoices"
assert rule["availablePermissions"] == ["inRole:roles/storage.objectViewer"]
assert "tenant-a/" in rule["availabilityCondition"]["expression"]

ev = BoundaryEvaluator(boundary)
assert ev.allows("gs://acme-invoices/tenant-a/2026-01.pdf", "storage.objects.get")
assert not ev.allows("gs://acme-invoices/tenant-b/2026-01.pdf", "storage.objects.get")
assert not ev.allows("gs://acme-invoices/tenant-a/2026-01.pdf", "storage.objects.delete")
assert not ev.allows("gs://other-bucket/tenant-a/x", "storage.objects.get")
print(json.dumps(rule, indent=2))

**In one sentence:** "Delegation is exchange, not forwarding: `sub` is the user, `act` is
the agent, scopes intersect, each hop nests another `act`, and the token is bound to the caller
by certificate or DPoP key so replay fails. For storage I downscope further with a Credential
Access Boundary."